Este notebook descarga el dataset [EuroSAT (RGB)](https://www.kaggle.com/datasets/ryanholbrook/eurosat) desde Kaggle y guarda las imágenes en `data/EuroSAT/<clase>/*.jpg`.


La carpeta `data/` está en el `.gitignore`, por lo que se debe de correr este codigo para descargar la base

In [ ]:
import struct
from pathlib import Path

import kagglehub


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT_DIR = ROOT / "data" / "EuroSAT"

path = Path(kagglehub.dataset_download("ryanholbrook/eurosat"))
print("Dataset en caché:", path)

El dataset de Kaggle viene como un archivo TFRecord (imágenes JPEG + etiqueta). Se lee con un parser mínimo para no depender de TensorFlow.

In [ ]:
def read_varint(buf, pos):
    result = shift = 0
    while True:
        b = buf[pos]; pos += 1
        result |= (b & 0x7F) << shift
        if not b & 0x80:
            return result, pos
        shift += 7

def parse_fields(buf):
    """Itera (número_de_campo, tipo_wire, valor) de un mensaje protobuf."""
    pos = 0
    while pos < len(buf):
        key, pos = read_varint(buf, pos)
        field, wire = key >> 3, key & 7
        if wire == 0:
            val, pos = read_varint(buf, pos)
        elif wire == 2:
            n, pos = read_varint(buf, pos)
            val, pos = buf[pos:pos + n], pos + n
        else:
            raise ValueError(f"wire type no soportado: {wire}")
        yield field, wire, val

def parse_example(record):
    """tf.train.Example -> {nombre: bytes | int}"""
    out = {}
    for _, _, features in parse_fields(record):                 # Example.features
        for _, _, entry in parse_fields(features):              # map<string, Feature>
            key = value = None
            for f, _, v in parse_fields(entry):
                if f == 1: key = v.decode()
                else: value = v
            for f, wire, v in parse_fields(value):              # Feature
                if f == 1:                                      # BytesList
                    out[key] = next(parse_fields(v))[2]
                elif f == 3:                                    # Int64List
                    first = next(parse_fields(v))
                    out[key] = first[2] if first[1] == 0 else read_varint(first[2], 0)[0]
    return out

def iter_tfrecord(file):
    with open(file, "rb") as f:
        while header := f.read(12):                             # longitud (8) + crc (4)
            (length,) = struct.unpack("<Q", header[:8])
            data = f.read(length)
            f.read(4)                                           # crc de los datos
            yield data

In [ ]:
tf_dir = next(path.rglob("label.labels.txt")).parent
labels = (tf_dir / "label.labels.txt").read_text().split()
shards = sorted(tf_dir.glob("*.tfrecord*"))

if OUT_DIR.exists() and any(OUT_DIR.rglob("*.jpg")):
    print("Las imágenes ya existen en", OUT_DIR, "- no se vuelve a descargar.")
else:
    for name in labels:
        (OUT_DIR / name).mkdir(parents=True, exist_ok=True)
    n = 0
    for shard in shards:
        for rec in iter_tfrecord(shard):
            ex = parse_example(rec)
            fname = Path(ex["filename"].decode()).stem
            (OUT_DIR / labels[ex["label"]] / f"{fname}.jpg").write_bytes(ex["image"])
            n += 1
    print(f"{n} imágenes guardadas en {OUT_DIR}")

Verificación: conteo por clase y una imagen de ejemplo.

In [ ]:
from PIL import Image

for d in sorted(OUT_DIR.iterdir()):
    print(f"{d.name:22s} {len(list(d.glob('*.jpg')))}")

sample = next(OUT_DIR.rglob("*.jpg"))
img = Image.open(sample)
print(sample.name, img.size, img.mode)
img